# 🚀 MultiModal RAG — Cloud Ingestion (Google Colab)

**Run this notebook on Colab to ingest PDFs into Qdrant Cloud.**
Your local machine then queries the same Qdrant Cloud collection for retrieval.

| Step | Where | Tool |
|------|-------|------|
| PDF Parsing | ☁️ Colab GPU | Ollama + glm-ocr |
| Image Captioning | ☁️ Colab | Groq (free) |
| Embeddings | ☁️ Colab GPU | sentence-transformers |
| Vector Storage | ☁️ Qdrant Cloud | cloud.qdrant.io (free) |
| Retrieval + Reranking | 🖥️ Local | BGE + FastAPI |

### Prerequisites
1. **Qdrant Cloud** free account → [cloud.qdrant.io](https://cloud.qdrant.io) (no credit card)
2. **Groq** free API key → [console.groq.com](https://console.groq.com)
3. Runtime → **Runtime > Change runtime type → T4 GPU**

> ⚠️ Fill in your keys in **Cell 1** before running anything else.

## 🔑 Cell 1 — Secrets & Config

Fill in your Qdrant Cloud URL, API key, and Groq API key below.

In [ ]:
import os

# ── FILL THESE IN ────────────────────────────────────────────────────────────
QDRANT_URL        = 'https://YOUR-CLUSTER.qdrant.io:6333'  # from cloud.qdrant.io
QDRANT_API_KEY    = 'your-qdrant-api-key'
QDRANT_COLLECTION = 'documents'

GROQ_API_KEY      = 'gsk_your_groq_key'    # console.groq.com → free
GROQ_VISION_MODEL = 'qwen/qwen3.6-27b'
GROQ_TEXT_MODEL   = 'meta/llama-3.3-70b-versatile'

EMBEDDING_MODEL   = 'all-MiniLM-L6-v2'   # 384d, ~22 MB
EMBEDDING_DIMS    = 384
# ─────────────────────────────────────────────────────────────────────────────

# Validate
assert 'YOUR-CLUSTER' not in QDRANT_URL, '❌ Set your Qdrant Cloud URL above'
assert not QDRANT_API_KEY.startswith('your-'), '❌ Set your Qdrant API key above'
assert not GROQ_API_KEY.startswith('gsk_your'), '❌ Set your Groq API key above'

# Export so submodules pick them up
os.environ.update({
    'QDRANT_URL': QDRANT_URL,
    'QDRANT_API_KEY': QDRANT_API_KEY,
    'QDRANT_COLLECTION_NAME': QDRANT_COLLECTION,
    'GROQ_API_KEY': GROQ_API_KEY,
    'GROQ_VISION_MODEL': GROQ_VISION_MODEL,
    'GROQ_TEXT_MODEL': GROQ_TEXT_MODEL,
    'EMBEDDING_PROVIDER': 'local',
    'EMBEDDING_MODEL': EMBEDDING_MODEL,
    'EMBEDDING_DIMENSIONS': str(EMBEDDING_DIMS),
    'PARSER_BACKEND': 'ollama',
    'RERANKER_BACKEND': 'bge',
    'IMAGE_CAPTION_ENABLED': 'true',
    'LOG_LEVEL': 'INFO',
})

print('✅ Config set')
print(f'   Qdrant  : {QDRANT_URL}')
print(f'   Groq    : {GROQ_API_KEY[:12]}...')
print(f'   Embed   : {EMBEDDING_MODEL} ({EMBEDDING_DIMS}d)')

## 📦 Cell 2 — Install Dependencies

Clones the repo and installs all required packages. Takes ~2 minutes on first run.

In [ ]:
import subprocess, sys

def run(cmd, **kw):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if r.stdout: print(r.stdout[-2000:])
    if r.returncode != 0:
        print('STDERR:', r.stderr[-1000:])
        raise RuntimeError(f'Command failed: {cmd}')
    return r

# Clone the project
import os
if not os.path.exists('/content/multi-modal-rag'):
    run('git clone https://github.com/YOUR_USERNAME/multi-modal-rag /content/multi-modal-rag')
    print('✅ Repo cloned')
else:
    print('✅ Repo already present')

os.chdir('/content/multi-modal-rag')
sys.path.insert(0, '/content/multi-modal-rag/src')

# Install uv for fast package management
run('pip install uv -q')

# Install project with GPU extras
run('uv pip install -e ".[local-embed,bge,layout]" --system -q')

print('✅ All packages installed')

## 🦙 Cell 3 — Install Ollama & Pull glm-ocr

Installs Ollama on Colab, starts the server, and downloads the glm-ocr model (~2.2 GB).

In [ ]:
import subprocess, time, urllib.request

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout[-1000:] if r.stdout else '')
    if r.returncode != 0: print('STDERR:', r.stderr[-500:])
    return r

# Install Ollama
print('Installing Ollama...')
run('curl -fsSL https://ollama.com/install.sh | sh')

# Start server in background
proc = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print('Ollama server starting...')
time.sleep(3)

# Verify
try:
    urllib.request.urlopen('http://localhost:11434')
    print('✅ Ollama server running')
except:
    print('⚠ Server not up yet, waiting...')
    time.sleep(5)

# Pull glm-ocr (2.2 GB — takes 2-5 min on Colab)
print('Pulling glm-ocr model (~2.2 GB)...')
run('ollama pull glm-ocr:latest')

# Confirm
r = run('ollama list')
print('✅ glm-ocr ready' if 'glm-ocr' in r.stdout else '✗ glm-ocr not found — check output above')

## 📂 Cell 4 — Upload your PDF

Upload a PDF from your computer into Colab, or mount Google Drive to use existing files.

In [ ]:
from google.colab import files as colab_files
from pathlib import Path
import os

# ── Option A: Upload from local computer ─────────────────────────────────────
print('Select a PDF to upload (or skip and use Option B below):')
uploaded = colab_files.upload()

if uploaded:
    PDF_PATH = Path(list(uploaded.keys())[0])
    print(f'✅ Uploaded: {PDF_PATH.name} ({PDF_PATH.stat().st_size / 1024:.1f} KB)')
else:
    # ── Option B: Mount Google Drive ─────────────────────────────────────────
    # from google.colab import drive
    # drive.mount('/content/drive')
    # PDF_PATH = Path('/content/drive/MyDrive/your_document.pdf')
    print('No file uploaded. Set PDF_PATH manually below.')
    PDF_PATH = Path('/content/your_document.pdf')  # change this

if not PDF_PATH.exists():
    print(f'✗ File not found: {PDF_PATH}')
else:
    print(f'📄 Will parse: {PDF_PATH.name}')

## 📄 Cell 5 — Parse PDF via Ollama + glm-ocr

First parse takes 30-60s (model warm-up), subsequent pages are faster.

In [ ]:
import sys, os
sys.path.insert(0, '/content/multi-modal-rag/src')
os.chdir('/content/multi-modal-rag')

from doc_parser.config import get_settings, configure_logging
from doc_parser.pipeline import DocumentParser

configure_logging('INFO')
settings = get_settings()

print(f'Parsing {PDF_PATH.name} via Ollama glm-ocr...')
parser = DocumentParser()
result = parser.parse_file(PDF_PATH)

print(f'✅ Parsed: {len(result.pages)} pages, {result.total_elements} elements')
for p in result.pages:
    labels = set(e.label for e in p.elements)
    print(f'  Page {p.page_num}: {len(p.elements)} elements — {labels}')

## 🧩 Cell 6 — Structure-Aware Chunking

In [ ]:
from collections import Counter
from doc_parser.chunker import document_aware_chunking

pages = [(p.page_num, p.elements) for p in result.pages]
all_chunks = document_aware_chunking(pages, source_file=PDF_PATH.name)

counts = Counter(c.modality for c in all_chunks)
print(f'✅ {len(all_chunks)} chunks created: {dict(counts)}')
for c in all_chunks[:3]:
    print(f'  [{c.chunk_id}] {c.modality} | {c.text[:100]}...')

## 🖼️ Cell 7 — Caption Images/Tables via Groq

Vision model: `qwen/qwen3.6-27b` (images) | Text model: `meta/llama-3.3-70b-versatile` (tables/formulas)

In [ ]:
from doc_parser.ingestion.image_captioner import enrich_chunks

non_text = [c for c in all_chunks if c.modality != 'text']
print(f'Captioning {len(non_text)} non-text chunks (images, tables, formulas) via Groq...')

all_chunks = await enrich_chunks(
    all_chunks,
    pdf_path=PDF_PATH,
    settings=settings,
    max_concurrent=3,   # Colab has better bandwidth — higher concurrency OK
)

captioned = [c for c in all_chunks if c.caption]
print(f'✅ {len(captioned)} chunks captioned')
for c in captioned[:2]:
    print(f'  [{c.modality}] {c.caption[:150]}')

## 🔢 Cell 8 — Embed (sentence-transformers, GPU-accelerated)

On a T4 GPU, embedding 100 chunks takes ~2 seconds vs ~20 seconds on CPU.

In [ ]:
import torch
from doc_parser.ingestion.embedder import get_embedder, embed_chunks

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# Force GPU for sentence-transformers if available
os.environ['SENTENCE_TRANSFORMERS_DEVICE'] = device

embedder = get_embedder(settings)
print(f'Embedding {len(all_chunks)} chunks with [{settings.embedding_model}] on {device}...')

dense, sparse = await embed_chunks(all_chunks, embedder, settings)

print(f'✅ Dense : {len(dense)} vectors × {len(dense[0])}d')
print(f'   Sparse: {len(sparse)} BM25 vectors')

## 🗄️ Cell 9 — Upsert to Qdrant Cloud

Writes all chunks to your Qdrant Cloud collection. Both Colab and your local machine can now query it!

In [ ]:
from doc_parser.ingestion.vector_store import QdrantDocumentStore

print(f'Connecting to Qdrant Cloud: {settings.qdrant_url}')
store = QdrantDocumentStore(settings)

# overwrite=False means it skips creation if collection already exists
await store.create_collection(overwrite=False)

upserted = await store.upsert_chunks(all_chunks, dense, sparse)
print(f'✅ Upserted {upserted} points → collection "{settings.qdrant_collection_name}"')
print(f'   Dashboard: {settings.qdrant_url}/dashboard')
print()
print('📋 LOCAL MACHINE SETUP:')
print(f'   Set in your .env on local:')
print(f'   QDRANT_URL={QDRANT_URL}')
print(f'   QDRANT_API_KEY={QDRANT_API_KEY}')
print(f'   EMBEDDING_DIMENSIONS={EMBEDDING_DIMS}')
print(f'   Then run Cells 0-2 + Cell 10 in 01_quickstart.ipynb to search!')

## 🔍 Cell 10 — Quick Search Verification (Optional)

Run a quick search from Colab itself to verify everything was stored correctly.

In [ ]:
QUERY = 'What is the main contribution of this paper?'

embedder2 = get_embedder(settings)
candidates = await store.search(
    query_text=QUERY,
    embedder=embedder2,
    settings=settings,
    top_k=5,
)

print(f'Query: "{QUERY}"')
print(f'Retrieved {len(candidates)} results from Qdrant Cloud:\n')
for i, r in enumerate(candidates, 1):
    print(f'[{i}] modality={r.get("modality")}  page={r.get("page")}')
    print(f'    {r.get("text","")[:200]}')
    print()